# Preprocessing audit

Compare retention, exact-record removal, and target averaging only inside training folds. Validation keeps all original observations. Audit invalid-value rules and agreeing-address recovery. The source lease year is masked in the recovery variant; original invalid flags remain visible.

Run notebooks 06–10 in order. Existing notebooks 01–05 and processed datasets are read-only inputs. Results are written to `experiments/results/`. Scores use individual rents in SGD; tuning folds are July–December 2023 and January–June 2024. No final-block scores are used in model or feature selection.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = Path.cwd() if (Path.cwd() / "data/train.csv").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from experiments import rental_experiments as exp


## Invalid-value definitions

The legacy lease age is invalid when rental year minus lease commencement year is below 0 or above 100; only the derived age was masked. HDB maximum floor/completion year values at or below 0 are invalid. Building ages below 0 or above 100 are masked. The repair variant additionally masks invalid source years and recovers only from valid, agreeing training addresses, then recomputes ages. Remaining numeric gaps use training medians. Raw completeness does not imply derived completeness.

In [2]:
scores = exp.audit()
display(pd.Series(pd.read_json(exp.OUT / "data_audit.json", typ="series")))
display(pd.read_csv(exp.OUT / "invalid_values.csv"))
display(pd.read_csv(exp.OUT / "block_recovery_audit.csv"))
display(scores)
display(exp.settings())

ridge keep_all median base {'alpha': 10} 2023_H2: RMSE=597.814; 1.1s


ridge keep_all median base {'alpha': 10} 2024_H1: RMSE=603.793; 1.2s


ridge keep_all block_recovery base {'alpha': 10} 2023_H2: RMSE=597.807; 1.2s


ridge keep_all block_recovery base {'alpha': 10} 2024_H1: RMSE=603.809; 1.4s


ridge exact_only median base {'alpha': 10} 2023_H2: RMSE=597.809; 1.2s


ridge exact_only median base {'alpha': 10} 2024_H1: RMSE=603.787; 1.5s


... (8 outputs left out)

,split,source,invalid_rows,affected_addresses
0,train,LEASE_COMMENCE_DATE,1,1
1,train,HDB_MAX_FLOOR,159,11
2,train,HDB_YEAR_COMPLETED,159,11
3,train,LEASE_AGE_AT_RENT,1,1
4,train,HDB_BUILDING_AGE_AT_RENT,159,11
5,test,LEASE_COMMENCE_DATE,0,0
6,test,HDB_MAX_FLOOR,0,0
7,test,HDB_YEAR_COMPLETED,0,0
8,test,LEASE_AGE_AT_RENT,0,0
9,test,HDB_BUILDING_AGE_AT_RENT,0,0


,fold,source,invalid_before,recovered,remaining_missing,agreeing_training_addresses
0,2023_H2,LEASE_COMMENCE_DATE,0,0,0,9170
1,2023_H2,HDB_MAX_FLOOR,9,0,9,9159
2,2023_H2,HDB_YEAR_COMPLETED,9,0,9,9159
3,2024_H1,LEASE_COMMENCE_DATE,0,0,0,9272
4,2024_H1,HDB_MAX_FLOOR,0,0,0,9261
5,2024_H1,HDB_YEAR_COMPLETED,0,0,0,9261


,model,policy,imputation,group,params,fold,RMSE,MAE,bias_actual_minus_predicted,train_rows,evaluation_rows,seconds
0,ridge,keep_all,median,base,"{""alpha"": 10}",2023_H2,597.813735,461.805151,-72.803507,87937,17900,1.122118
1,ridge,keep_all,median,base,"{""alpha"": 10}",2024_H1,603.792881,439.490253,-300.285165,105837,18179,1.223182
2,ridge,keep_all,block_recovery,base,"{""alpha"": 10}",2023_H2,597.806985,461.813783,-72.694134,87937,17900,1.151772
3,ridge,keep_all,block_recovery,base,"{""alpha"": 10}",2024_H1,603.809265,439.507799,-300.321325,105837,18179,1.414311
4,ridge,exact_only,median,base,"{""alpha"": 10}",2023_H2,597.808612,461.819253,-72.669266,87935,17900,1.192597
5,ridge,exact_only,median,base,"{""alpha"": 10}",2024_H1,603.787126,439.501485,-300.240194,105798,18179,1.457274
6,ridge,exact_only,block_recovery,base,"{""alpha"": 10}",2023_H2,597.807312,461.789271,-72.798998,87935,17900,1.236078
7,ridge,exact_only,block_recovery,base,"{""alpha"": 10}",2024_H1,603.776674,439.484899,-300.224666,105798,18179,1.489188
8,ridge,collapse_mean,median,base,"{""alpha"": 10}",2023_H2,599.264502,462.235278,-80.834751,75329,17900,1.502302
9,ridge,collapse_mean,median,base,"{""alpha"": 10}",2024_H1,608.666153,443.726427,-308.824109,90671,18179,1.818546


{'policy': 'keep_all',
 'imputation': 'block_recovery',
 'selection_rule': 'Within 1 SGD of best early-fold mean RMSE, prefer keep_all then exact_only; prefer masking/recovery when within 0.1 SGD of best imputation.',
 'mean_RMSE': 600.8081251689713}

## Interpretation

Read per-fold scores before interpreting the mean. A lower score does not establish causality. Explain temporal drift, duplicate uncertainty, snapshot limitations, and computational cost in the report. Hyperparameters and selections are saved alongside results.

References: [CatBoost fit API](https://catboost.ai/docs/en/concepts/python-reference_catboostregressor_fit); [scikit-learn MLPRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPRegressor.html).